# Dictionary Views — Advanced Tutorial Problems with Solutions

This notebook is a **tutorial-style problem set** about:

- `dict.keys()`
- `dict.values()`
- `dict.items()`
- dynamic/live dictionary views
- iteration behavior
- safe and unsafe mutation
- snapshots
- set-like operations
- mapping proxies
- ordering
- performance
- debugging and API design

Unlike a compact exercise sheet, each problem is broken into logical stages.

For most problems we will use this rhythm:

1. **Situation** — understand the problem.
2. **Prediction** — reason before running code.
3. **Experiment** — execute a small example.
4. **Observation** — explain exactly what happened.
5. **Solution** — implement the robust version.
6. **Takeaway** — extract the reusable rule.

The examples are intentionally different from basic textbook demonstrations and focus on reasoning patterns that appear in real programs.


## Notebook conventions

A few principles will be used throughout:

- We prefer readable code over clever one-liners.
- We use assertions to verify important claims.
- We distinguish **changing a value** from **changing dictionary structure**.
- When a structural mutation is required, we make the iteration domain stable first.
- Timing experiments are treated as demonstrations, not universal performance laws.


In [1]:
from __future__ import annotations

from collections.abc import Mapping
from copy import deepcopy
from timeit import timeit
from types import MappingProxyType
from typing import Any, Callable

print("Setup complete.")


Setup complete.


# Problem 1 — A View Captured Before the Data Exists

## Situation

A plugin manager is created before all plugins are registered.

We capture:

```python
plugin_names = plugins.keys()
```

At that moment the dictionary is empty.

Later, several plugins are registered.

### Question

Must we call `plugins.keys()` again to see the new plugins?


## Prediction

Before running the next cell, decide which statement is correct:

- **A.** `plugin_names` stays empty forever.
- **B.** `plugin_names` automatically reflects later changes.
- **C.** It depends on the number of inserted keys.

A dictionary view is not a copied list. It remains connected to its dictionary.


In [2]:
plugins = {}
plugin_names = plugins.keys()

print("Immediately after capture:", plugin_names)

plugins["csv"] = object()
plugins["json"] = object()
plugins["xml"] = object()

print("After registration:", plugin_names)

assert set(plugin_names) == {"csv", "json", "xml"}


Immediately after capture: dict_keys([])
After registration: dict_keys(['csv', 'json', 'xml'])


## Observation

The same `plugin_names` object now exposes three keys.

No reassignment was necessary.

This is the defining property of a dictionary view: it is a **live interface** to the current mapping.


## Takeaway

If you need **current state**, a view is often ideal.

If you need **state as it existed at one moment**, use a snapshot such as:

```python
tuple(plugins)
list(plugins)
set(plugins)
```


# Problem 2 — A Historical Audit Bug

## Situation

An audit function stores this value at 09:00:

```python
morning_users = users.keys()
```

At 17:00 it prints `morning_users`, expecting to show the users that existed at 09:00.

During the day, users were added and removed.

### Task

Explain why this audit is logically incorrect and repair it.


In [3]:
users = {
    "alice": "active",
    "bob": "active",
}

morning_users = users.keys()

# Simulate later changes.
users["carol"] = "active"
del users["bob"]

print("Audit result:", morning_users)


Audit result: dict_keys(['alice', 'carol'])


## What went wrong?

The programmer wanted **historical state**, but selected a data structure designed for **live state**.

The issue is not that the view is malfunctioning.

The issue is that the view is doing exactly what views are designed to do.


## Step 1 — Choose a snapshot representation

For an audit, order may matter.

A tuple is a good choice because:

- it freezes the key sequence,
- it is immutable,
- it communicates snapshot intent clearly.


In [4]:
users = {
    "alice": "active",
    "bob": "active",
}

morning_users = tuple(users)

users["carol"] = "active"
del users["bob"]

print("Morning snapshot:", morning_users)
print("Current users:", tuple(users))

assert morning_users == ("alice", "bob")
assert tuple(users) == ("alice", "carol")


Morning snapshot: ('alice', 'bob')
Current users: ('alice', 'carol')


## Takeaway

The important design question is not:

> “Should I use `.keys()` or a list?”

It is:

> “Do I want a **live window** or a **stable historical snapshot**?”


# Problem 3 — Which Mutations Are Structural?

## Situation

Consider this dictionary:

```python
inventory = {"A": 4, "B": 7, "C": 10}
```

We perform four possible operations while iterating:

1. replace `inventory["A"]`,
2. append to a list stored as a value,
3. add key `"D"`,
4. delete key `"B"`.

### Task

Classify each operation as:

- value-only mutation,
- nested-object mutation,
- structural dictionary mutation.


## Reasoning

A dictionary iterator fundamentally cares about the mapping's structure.

Changing which keys exist changes that structure.

Replacing the value attached to an already-existing key does not change the key domain.

Mutating a mutable object *inside* a value is even further removed from the dictionary's structure.


In [5]:
inventory = {
    "A": 4,
    "B": 7,
    "C": 10,
}

for key in inventory:
    inventory[key] *= 10

print(inventory)

assert inventory == {
    "A": 40,
    "B": 70,
    "C": 100,
}


{'A': 40, 'B': 70, 'C': 100}


## Observation

Replacing values for existing keys is safe in this pattern because the dictionary remains the same size and has the same keys.


## Takeaway

A useful first approximation is:

- **same keys, different values** → generally safe during iteration;
- **added/removed keys** → treat as structural mutation and use a safer pattern.


# Problem 4 — Why the Error Appears on the Next Iteration Step

## Situation

A common surprise is that this line can execute:

```python
del d[key]
```

and the `RuntimeError` appears only when the loop tries to continue.

Let's isolate that behavior.


In [6]:
d = {"x": 1, "y": 2, "z": 3}
iterator = iter(d)

first = next(iterator)
print("First key:", first)

del d[first]
print("Deletion succeeded.")
print("Dictionary is now:", d)

try:
    next(iterator)
except RuntimeError as exc:
    print("Iterator detected the structural change:", exc)


First key: x
Deletion succeeded.
Dictionary is now: {'y': 2, 'z': 3}
Iterator detected the structural change: dictionary changed size during iteration


## Observation

The deletion operation itself is valid.

What becomes invalid is the assumption held by the already-created iterator about the dictionary's structure.


## Takeaway

When debugging mutation errors, distinguish:

- **the mutation operation**, and
- **the iterator's next attempt to proceed**.

This explains why traceback locations can be unintuitive.


# Problem 5 — Remove Expired Sessions

## Situation

We have session expiry timestamps represented by integers.

We want to remove all expired sessions.

A tempting implementation is:

```python
for session, expiry in sessions.items():
    if expiry < now:
        del sessions[session]
```

We need a robust design.


## Step 1 — Separate discovery from mutation

First, determine which session IDs should disappear.

During this discovery phase, do not change the dictionary structure.


In [7]:
sessions = {
    "s1": 120,
    "s2": 80,
    "s3": 150,
    "s4": 60,
}

now = 100

expired = [
    session_id
    for session_id, expiry in sessions.items()
    if expiry < now
]

print("Expired:", expired)


Expired: ['s2', 's4']


## Step 2 — Perform the structural mutation afterward

Now that the iteration phase has ended, deletion is straightforward.


In [8]:
for session_id in expired:
    del sessions[session_id]

print("Remaining sessions:", sessions)

assert sessions == {
    "s1": 120,
    "s3": 150,
}


Remaining sessions: {'s1': 120, 's3': 150}


## Why this style is valuable

The code has two clearly separated responsibilities:

1. decide what should change,
2. apply the changes.

That separation is often easier to test, log, and debug than mutating from inside the discovery loop.


# Problem 6 — Snapshot the Keys, but Read the Latest Values

## Situation

Sometimes we want stable keys but **fresh values**.

Imagine we snapshot the IDs of active jobs. Before processing begins, job priorities may still be updated.

We want:

- the same original job IDs,
- the latest priority for each ID.


## Step 1 — Snapshot only the keys

A key-only snapshot freezes the traversal domain without freezing values.


In [9]:
jobs = {
    "job-A": 1,
    "job-B": 2,
    "job-C": 3,
}

batch_ids = list(jobs)

# Priorities change after the batch IDs are chosen.
jobs["job-B"] = 200
jobs["job-C"] = 300

for job_id in batch_ids:
    print(job_id, jobs[job_id])


job-A 1
job-B 200
job-C 300


## Observation

The key sequence is stable because `batch_ids` is a list.

The values are current because they are looked up from `jobs` during processing.


## Takeaway

Choose the snapshot level deliberately:

- `list(d)` → stable keys, values may still be fetched live;
- `list(d.items())` → stable sequence of captured key/value references;
- `deepcopy(...)` → stronger isolation for nested mutable data.


# Problem 7 — The Shallow Item Snapshot

## Situation

We snapshot this dictionary:

```python
teams = {
    "red": ["Ada", "Grace"],
    "blue": ["Linus"]
}
```

using:

```python
snapshot = tuple(teams.items())
```

Then someone appends a name to `teams["red"]`.

Will the snapshot change?


In [10]:
teams = {
    "red": ["Ada", "Grace"],
    "blue": ["Linus"],
}

snapshot = tuple(teams.items())

teams["red"].append("Guido")

print("Dictionary:", teams)
print("Snapshot:", snapshot)

assert snapshot[0][1] == ["Ada", "Grace", "Guido"]


Dictionary: {'red': ['Ada', 'Grace', 'Guido'], 'blue': ['Linus']}
Snapshot: (('red', ['Ada', 'Grace', 'Guido']), ('blue', ['Linus']))


## Explanation

The tuple froze the outer sequence of item pairs.

It did **not** recursively clone each value.

The list object stored under `"red"` is shared.


## Step 2 — Create an isolated historical snapshot

If independent nested data is required, a deep copy is one possible tool.


In [11]:
teams = {
    "red": ["Ada", "Grace"],
    "blue": ["Linus"],
}

isolated = deepcopy(tuple(teams.items()))

teams["red"].append("Guido")

print("Current:", teams)
print("Isolated:", isolated)

assert isolated[0][1] == ["Ada", "Grace"]


Current: {'red': ['Ada', 'Grace', 'Guido'], 'blue': ['Linus']}
Isolated: (('red', ['Ada', 'Grace']), ('blue', ['Linus']))


## Takeaway

“Snapshot” is not automatically synonymous with “deep copy.”

Always ask which levels of the object graph must be isolated.


# Problem 8 — Key Views as Mathematical Sets

## Situation

Two API versions support different endpoint names.

We want to understand the relationship between the two key domains without copying them into sets first.


In [12]:
api_v1 = {
    "users": "/v1/users",
    "orders": "/v1/orders",
    "search": "/v1/search",
    "legacy": "/v1/legacy",
}

api_v2 = {
    "users": "/v2/users",
    "orders": "/v2/orders",
    "search": "/v2/search",
    "metrics": "/v2/metrics",
}

v1 = api_v1.keys()
v2 = api_v2.keys()

print("Common:", v1 & v2)
print("Removed in v2:", v1 - v2)
print("Added in v2:", v2 - v1)
print("Exactly one version:", v1 ^ v2)


Common: {'users', 'search', 'orders'}
Removed in v2: {'legacy'}
Added in v2: {'metrics'}
Exactly one version: {'legacy', 'metrics'}


## Why no `set(...)` conversion was needed

`dict_keys` implements set-like behavior.

This is especially convenient for:

- schema comparison,
- configuration validation,
- feature comparison,
- migration analysis,
- missing/extra-field detection.


## Mini challenge

What expression answers:

> “Are all v1 endpoint names still present in v2?”

Use a subset relation.


In [13]:
all_v1_preserved = api_v1.keys() <= api_v2.keys()

print(all_v1_preserved)
assert all_v1_preserved is False


False


# Problem 9 — Validate Required and Allowed Fields

## Situation

A request payload must satisfy two rules:

1. it must contain all required fields,
2. it must not contain fields outside the allowed schema.

Required:

```python
{"username", "email"}
```

Allowed:

```python
{"username", "email", "display_name", "timezone"}
```


## Step 1 — Compare the payload's key view directly

The key view is already the natural representation of “which fields exist?”


In [14]:
required = {"username", "email"}
allowed = {"username", "email", "display_name", "timezone"}

payload = {
    "username": "ada",
    "display_name": "Ada",
    "admin": True,
}

actual = payload.keys()

missing = required - actual
unexpected = actual - allowed

print("Missing:", missing)
print("Unexpected:", unexpected)


Missing: {'email'}
Unexpected: {'admin'}


## Step 2 — Build a reusable validator

The validator should report both categories instead of stopping at the first problem.


In [15]:
def validate_fields(
    payload: Mapping[str, Any],
    required: set[str],
    allowed: set[str],
) -> dict[str, set[str]]:
    actual = payload.keys()

    return {
        "missing": required - actual,
        "unexpected": actual - allowed,
    }


result = validate_fields(payload, required, allowed)
print(result)

assert result == {
    "missing": {"email"},
    "unexpected": {"admin"},
}


{'missing': {'email'}, 'unexpected': {'admin'}}


## Takeaway

Views let the code speak in terms of the real abstraction:

> compare domains of keys

rather than:

> convert several things to containers, then compare them.


# Problem 10 — Exact Assignment Differences with Item Views

## Situation

A deployment maps machines to release versions.

We want to know which exact machine/version assignments changed between two moments.

A key comparison alone is insufficient because the same key may remain while its value changes.


In [16]:
before = {
    "web-1": "1.4.0",
    "web-2": "1.4.0",
    "worker-1": "1.3.9",
}

after = {
    "web-1": "1.4.0",
    "web-2": "1.4.1",
    "worker-2": "1.4.1",
}

stable_pairs = before.items() & after.items()
removed_pairs = before.items() - after.items()
added_pairs = after.items() - before.items()

print("Stable exact assignments:", stable_pairs)
print("Assignments no longer present:", removed_pairs)
print("New exact assignments:", added_pairs)


Stable exact assignments: {('web-1', '1.4.0')}
Assignments no longer present: {('web-2', '1.4.0'), ('worker-1', '1.3.9')}
New exact assignments: {('worker-2', '1.4.1'), ('web-2', '1.4.1')}


## Important interpretation

For `"web-2"`:

- the key exists before and after,
- but `("web-2", "1.4.0")` disappears,
- and `("web-2", "1.4.1")` appears.

Item-view set operations naturally describe exact pair-level changes.


## Caveat

Item-view set operations depend on hashability of the pairs involved.

Dictionary keys are hashable by definition.

Values are not required to be hashable.


# Problem 11 — An Item-View Operation Fails

## Situation

Now values are lists:

```python
left = {"group": ["a", "b"]}
right = {"group": ["a", "b"]}
```

The list values are unhashable.

Let's see how that affects item-view set operations.


In [17]:
left = {"group": ["a", "b"]}
right = {"group": ["a", "b"]}

try:
    common = left.items() & right.items()
    print(common)
except TypeError as exc:
    print("Expected failure:", exc)


Expected failure: unhashable type: 'list'


## Repair strategy

If pair-level set logic is truly required, transform values to a hashable representation that matches the intended semantics.

For lists where order matters, a tuple may be appropriate.


In [18]:
left_normalized = {
    key: tuple(value)
    for key, value in left.items()
}

right_normalized = {
    key: tuple(value)
    for key, value in right.items()
}

common = left_normalized.items() & right_normalized.items()

print(common)

assert common == {("group", ("a", "b"))}


{('group', ('a', 'b'))}


## Takeaway

Do not normalize merely to “make the error disappear.”

Choose a hashable representation only if it preserves the meaning your program needs.


# Problem 12 — Why `dict_values` Is Different

## Situation

Keys are unique.

Values are not.

Consider:

```python
scores = {"A": 10, "B": 10, "C": 20}
```

The value `10` appears twice.

This is one reason value views do not behave like key-set views.


In [19]:
scores = {
    "A": 10,
    "B": 10,
    "C": 20,
}

values = scores.values()

print(values)
print("10 is present:", 10 in values)
print("Number of values:", len(values))
print("Unique values:", set(values))


dict_values([10, 10, 20])
10 is present: True
Number of values: 3
Unique values: {10, 20}


## Observation

The value view preserves the mapping's current value sequence, including duplicates.

Converting it to a set changes the semantics by discarding multiplicity.


## Takeaway

Use:

```python
d.values()
```

when you want mapping values.

Use:

```python
set(d.values())
```

only when **unique-value set semantics** are actually intended and the values are hashable.


# Problem 13 — A Live Metrics Reader

## Situation

A monitoring object should always report statistics from the current dictionary.

Recomputing or copying the list of values every time is unnecessary.

We can intentionally retain a values view.


In [20]:
class LiveStatistics:
    def __init__(self, data: dict[str, float]) -> None:
        self._values = data.values()

    def count(self) -> int:
        return len(self._values)

    def total(self) -> float:
        return sum(self._values)

    def mean(self) -> float:
        count = self.count()
        return self.total() / count if count else 0.0


metrics = {
    "cpu": 20.0,
    "memory": 40.0,
}

stats = LiveStatistics(metrics)

print(stats.count(), stats.total(), stats.mean())


2 60.0 30.0


## Step 2 — Mutate the original mapping

The statistics object is not given the dictionary again.


In [21]:
metrics["disk"] = 60.0
metrics["cpu"] = 10.0

print(stats.count(), stats.total(), stats.mean())

assert stats.count() == 3
assert stats.total() == 110.0


3 110.0 36.666666666666664


## Takeaway

Live views can be a useful API feature when fresh state is explicitly desired.

The same feature becomes a bug when historical state is expected.

Intent matters.


# Problem 14 — A View Can Extend the Dictionary's Lifetime

## Situation

A view contains no copied collection of keys, but it still needs access to its dictionary.

Therefore, retaining the view also retains a reference to the dictionary.


In [22]:
import sys

data = {i: i * i for i in range(1000)}

before = sys.getrefcount(data)
view = data.keys()
after = sys.getrefcount(data)

print("Reference count before view:", before)
print("Reference count after view:", after)


Reference count before view: 2
Reference count after view: 3


## Interpretation

Exact reference counts are implementation details and may differ across runtimes.

The conceptual point is more important:

> the view must keep the underlying mapping reachable in order to remain live.


## Practical consequence

If you are archiving only a fixed list of keys, storing a snapshot may be a better ownership decision than storing a long-lived view.


# Problem 15 — Iterating over a Dictionary vs `.keys()`

## Situation

These loops look different:

```python
for key in d:
    ...

for key in d.keys():
    ...
```

For ordinary key iteration, they express essentially the same intent.


In [23]:
d = {
    "north": 1,
    "south": 2,
    "east": 3,
    "west": 4,
}

direct = [key for key in d]
through_view = [key for key in d.keys()]

print(direct)
print(through_view)

assert direct == through_view


['north', 'south', 'east', 'west']
['north', 'south', 'east', 'west']


## Style guidance

Prefer:

```python
for key in d:
    ...
```

for plain key iteration.

Use:

```python
d.keys()
```

when you specifically need the view object, especially for:

- set operations,
- passing the key view somewhere,
- storing a live key-domain reference.


# Problem 16 — Why `.items()` Usually Wins When You Need Both

## Situation

We want both key and value.

Two styles are possible:

```python
for key in d:
    value = d[key]
```

and:

```python
for key, value in d.items():
```

The second form expresses the requirement directly.


In [24]:
records = {
    i: i % 31
    for i in range(20_000)
}

def lookup_style(d: dict[int, int]) -> int:
    total = 0
    for key in d:
        total += key * d[key]
    return total


def items_style(d: dict[int, int]) -> int:
    total = 0
    for key, value in d.items():
        total += key * value
    return total


assert lookup_style(records) == items_style(records)


## Step 2 — Time both versions

The goal is not to prove a universal constant factor.

The goal is to see that avoiding explicit repeated lookup can matter.


In [25]:
t_lookup = timeit(lambda: lookup_style(records), number=200)
t_items = timeit(lambda: items_style(records), number=200)

print(f"lookup style: {t_lookup:.4f}s")
print(f"items style:  {t_items:.4f}s")
print(f"ratio:        {t_lookup / t_items:.2f}x")


lookup style: 0.4269s
items style:  0.6155s
ratio:        0.69x


## Best practice

Choose `.items()` first because it is:

- clearer,
- direct,
- often faster,
- less repetitive.


# Problem 17 — Normalize Values In Place

## Situation

A dictionary stores raw weights:

```python
{"A": 2, "B": 3, "C": 5}
```

We want each value divided by the total.

No keys are added or removed.

This is a perfect case for safe value replacement during iteration.


## Step 1 — Compute information that should remain stable

The original total should be calculated before values begin changing.


In [26]:
weights = {
    "A": 2,
    "B": 3,
    "C": 5,
}

total = sum(weights.values())
print("Original total:", total)


Original total: 10


## Step 2 — Replace each existing value

The key domain stays unchanged.


In [27]:
for key, value in weights.items():
    weights[key] = value / total

print(weights)

assert abs(sum(weights.values()) - 1.0) < 1e-12


{'A': 0.2, 'B': 0.3, 'C': 0.5}


## Takeaway

When updating existing values, it is often useful to iterate over `.items()` so the original value for that iteration is already available.


# Problem 18 — Mutating an Object Stored as a Value

## Situation

A dictionary maps projects to mutable task lists.

We want to append `"review"` to every list.

The dictionary itself does not gain or lose keys.


In [28]:
projects = {
    "compiler": ["parse", "optimize"],
    "website": ["design", "deploy"],
    "database": ["migrate"],
}

for project, tasks in projects.items():
    tasks.append("review")

print(projects)


{'compiler': ['parse', 'optimize', 'review'], 'website': ['design', 'deploy', 'review'], 'database': ['migrate', 'review']}


## Why this is different from structural mutation

The dictionary still has exactly the same key/value slots.

What changed is the internal state of objects referenced by those slots.

That is not the same operation as inserting or deleting dictionary keys.


## Takeaway

Always ask **which object is being mutated**:

- the dictionary structure,
- the value reference stored under a key,
- or a mutable object referenced by that value.


# Problem 19 — Safely Move Entries Between Dictionaries

## Situation

We have a source dictionary and want to move every record with priority at least 8 into another dictionary.

A direct `pop()` inside `for key, value in source.items()` would structurally mutate `source`.

We need a staged approach.


## Step 1 — Identify the keys to move


In [29]:
source = {
    "A": 3,
    "B": 8,
    "C": 5,
    "D": 10,
    "E": 9,
}

destination = {}

to_move = [
    key
    for key, priority in source.items()
    if priority >= 8
]

print(to_move)


['B', 'D', 'E']


## Step 2 — Move them after iteration has finished


In [30]:
for key in to_move:
    destination[key] = source.pop(key)

print("Source:", source)
print("Destination:", destination)

assert source == {"A": 3, "C": 5}
assert destination == {"B": 8, "D": 10, "E": 9}


Source: {'A': 3, 'C': 5}
Destination: {'B': 8, 'D': 10, 'E': 9}


## Takeaway

For structural changes, a small intermediate collection of keys is often the cleanest synchronization boundary between “inspect” and “mutate.”


# Problem 20 — Consuming a Dictionary with `popitem()`

## Situation

Sometimes we do not want to iterate over a live view at all.

We want to **consume** the dictionary until it is empty.

`popitem()` is designed for this type of destructive processing.


In [31]:
pending = {
    "compile": 4,
    "test": 7,
    "package": 2,
    "deploy": 9,
}

processed = []

while pending:
    name, cost = pending.popitem()
    processed.append((name, cost))

print("Processed:", processed)
print("Pending:", pending)

assert pending == {}


Processed: [('deploy', 9), ('package', 2), ('test', 7), ('compile', 4)]
Pending: {}


## Ordering detail

In modern Python, dictionaries preserve insertion order.

`popitem()` removes the most recently inserted pair, so this pattern behaves like LIFO consumption.


## When this pattern is appropriate

Use it when:

- destruction of the mapping is intentional,
- order is acceptable,
- no stable iteration view is required.

Do not use it merely to work around an iteration bug if the dictionary is supposed to remain intact.


# Problem 21 — Reverse Iteration

## Situation

Dictionary order is part of the language model in modern Python.

Views can participate in reverse iteration.

Let's verify all three views.


In [32]:
timeline = {
    "created": 1,
    "validated": 2,
    "approved": 3,
    "published": 4,
}

print("Reverse keys:")
print(list(reversed(timeline.keys())))

print("\nReverse values:")
print(list(reversed(timeline.values())))

print("\nReverse items:")
print(list(reversed(timeline.items())))


Reverse keys:
['published', 'approved', 'validated', 'created']

Reverse values:
[4, 3, 2, 1]

Reverse items:
[('published', 4), ('approved', 3), ('validated', 2), ('created', 1)]


## Use case

Reverse iteration can be useful when the newest inserted entries should be examined first without destroying the dictionary.


# Problem 22 — Renaming Keys Safely

## Situation

A mapping contains snake_case names.

We want uppercase keys.

This changes the key domain, so an in-place key rename during normal iteration is not a safe pattern.

Instead, stage a transformed mapping.


## Step 1 — Build the transformed result independently


In [33]:
source = {
    "first_name": "Ada",
    "last_name": "Lovelace",
    "birth_year": 1815,
}

transformed = {
    key.upper(): value
    for key, value in source.items()
}

print(transformed)


{'FIRST_NAME': 'Ada', 'LAST_NAME': 'Lovelace', 'BIRTH_YEAR': 1815}


## Step 2 — Decide whether the original object identity must be preserved

If callers hold references to the original dictionary and we want to mutate that same object, we can commit after staging.


In [34]:
source.clear()
source.update(transformed)

print(source)

assert source == {
    "FIRST_NAME": "Ada",
    "LAST_NAME": "Lovelace",
    "BIRTH_YEAR": 1815,
}


{'FIRST_NAME': 'Ada', 'LAST_NAME': 'Lovelace', 'BIRTH_YEAR': 1815}


## Takeaway

For structural transformations, **stage then commit** is often easier to reason about than modifying keys one at a time.


# Problem 23 — Detect Rename Collisions Before Commit

## Situation

Lowercasing these keys causes a collision:

```python
{"User": 1, "USER": 2}
```

A robust transformation should detect the problem before the original dictionary is touched.


In [35]:
def build_renamed_mapping(
    source: Mapping[str, Any],
    transform: Callable[[str], str],
) -> dict[str, Any]:
    result: dict[str, Any] = {}

    for old_key, value in source.items():
        new_key = transform(old_key)

        if new_key in result:
            raise ValueError(
                f"key collision: {old_key!r} -> {new_key!r}"
            )

        result[new_key] = value

    return result


## Step 2 — Test the failure path

The original must remain unchanged.


In [36]:
source = {
    "User": 1,
    "USER": 2,
}

before = source.copy()

try:
    renamed = build_renamed_mapping(source, str.lower)
except ValueError as exc:
    print("Rejected:", exc)

print("Original:", source)

assert source == before


Rejected: key collision: 'USER' -> 'user'
Original: {'User': 1, 'USER': 2}


## Takeaway

When a transformation can fail, postponing mutation gives us transaction-like behavior:

- validate first,
- commit second.


# Problem 24 — A Transactional Structural Transformation

## Situation

We now want to transform both keys and values.

Requirements:

- key transformation may fail,
- value transformation may fail,
- collisions are forbidden,
- the original mapping must remain unchanged unless everything succeeds.


## Step 1 — Build a staging dictionary

No structural mutation of the original occurs during validation.


In [37]:
def transform_transactionally(
    target: dict[Any, Any],
    key_transform: Callable[[Any], Any],
    value_transform: Callable[[Any, Any], Any],
) -> None:
    staged: dict[Any, Any] = {}

    for old_key, old_value in target.items():
        new_key = key_transform(old_key)
        new_value = value_transform(old_key, old_value)

        if new_key in staged:
            raise ValueError(
                f"duplicate transformed key: {new_key!r}"
            )

        staged[new_key] = new_value

    target.clear()
    target.update(staged)


## Step 2 — Successful commit


In [38]:
data = {
    "a": 2,
    "b": 4,
    "c": 6,
}

transform_transactionally(
    data,
    key_transform=str.upper,
    value_transform=lambda key, value: value ** 2,
)

print(data)

assert data == {
    "A": 4,
    "B": 16,
    "C": 36,
}


{'A': 4, 'B': 16, 'C': 36}


## Step 3 — Failed transformation

Now deliberately cause a collision.


In [39]:
data = {
    "A": 1,
    "a": 2,
}

before = data.copy()

try:
    transform_transactionally(
        data,
        key_transform=str.lower,
        value_transform=lambda key, value: value,
    )
except ValueError as exc:
    print("Failure:", exc)

assert data == before
print("Original preserved:", data)


Failure: duplicate transformed key: 'a'
Original preserved: {'A': 1, 'a': 2}


# Problem 25 — A Read-Only Live Mapping

## Situation

A library wants to expose current registry data to callers.

Requirements:

- callers should see future updates,
- callers should not mutate through the public object.

A copy would be read-independent but not live.

A mapping proxy provides a different semantic: **live but read-only through the proxy**.


In [40]:
class Registry:
    def __init__(self) -> None:
        self._data: dict[str, Any] = {}
        self._public = MappingProxyType(self._data)

    @property
    def public(self):
        return self._public

    def register(self, name: str, value: Any) -> None:
        self._data[name] = value


registry = Registry()
public = registry.public

registry.register("parser", "v1")
registry.register("renderer", "v2")

print(public)


{'parser': 'v1', 'renderer': 'v2'}


## Step 2 — Verify that updates are visible


In [41]:
registry.register("validator", "v3")

print(public)

assert public["validator"] == "v3"


{'parser': 'v1', 'renderer': 'v2', 'validator': 'v3'}


## Step 3 — Verify write protection through the proxy


In [42]:
try:
    public["parser"] = "modified"
except TypeError as exc:
    print("Write rejected:", exc)


Write rejected: 'mappingproxy' object does not support item assignment


## Takeaway

A copied dictionary and a mapping proxy solve different problems:

- **copy** → independent snapshot,
- **mapping proxy** → live read-only facade.


# Problem 26 — Access the Mapping Behind a View

## Situation

Modern dictionary view objects expose a `.mapping` attribute.

This provides a read-only mapping proxy associated with the view's dictionary.

Let's explore it.


In [43]:
settings = {
    "theme": "dark",
    "autosave": True,
}

keys = settings.keys()
readonly_mapping = keys.mapping

print(readonly_mapping)


{'theme': 'dark', 'autosave': True}


## Step 2 — Mutate the original dictionary

The proxy should remain live.


In [44]:
settings["language"] = "en"

print(readonly_mapping)

assert readonly_mapping["language"] == "en"


{'theme': 'dark', 'autosave': True, 'language': 'en'}


## Step 3 — Attempt to write through the proxy


In [45]:
try:
    readonly_mapping["theme"] = "light"
except TypeError as exc:
    print("Read-only:", exc)


Read-only: 'mappingproxy' object does not support item assignment


# Problem 27 — Dictionary Diff in Logical Stages

## Situation

We want a reusable configuration diff.

Given `old` and `new`, classify:

- added keys,
- removed keys,
- changed values,
- unchanged values.

Instead of writing one large comprehension, we will derive the result step by step.


## Step 1 — Compare the key domains


In [46]:
old = {
    "host": "db.internal",
    "port": 5432,
    "timeout": 10,
    "retries": 3,
}

new = {
    "host": "db.internal",
    "port": 6432,
    "timeout": 10,
    "ssl": True,
}

old_keys = old.keys()
new_keys = new.keys()

added_keys = new_keys - old_keys
removed_keys = old_keys - new_keys
common_keys = old_keys & new_keys

print("Added keys:", added_keys)
print("Removed keys:", removed_keys)
print("Common keys:", common_keys)


Added keys: {'ssl'}
Removed keys: {'retries'}
Common keys: {'host', 'port', 'timeout'}


## Step 2 — Split common keys by value equality


In [47]:
changed_keys = {
    key
    for key in common_keys
    if old[key] != new[key]
}

unchanged_keys = {
    key
    for key in common_keys
    if old[key] == new[key]
}

print("Changed:", changed_keys)
print("Unchanged:", unchanged_keys)


Changed: {'port'}
Unchanged: {'host', 'timeout'}


## Step 3 — Build a rich result


In [48]:
diff = {
    "added": {
        key: new[key]
        for key in added_keys
    },
    "removed": {
        key: old[key]
        for key in removed_keys
    },
    "changed": {
        key: (old[key], new[key])
        for key in changed_keys
    },
    "unchanged": {
        key: old[key]
        for key in unchanged_keys
    },
}

print(diff)


{'added': {'ssl': True}, 'removed': {'retries': 3}, 'changed': {'port': (5432, 6432)}, 'unchanged': {'host': 'db.internal', 'timeout': 10}}


## Takeaway

Breaking the problem into key-domain algebra first often makes the later value logic much easier to understand.


# Problem 28 — A Hidden Alias Causes a Mutation Bug

## Situation

This code appears to iterate one dictionary and modify another:

```python
current = data
other = data
```

But those variables are aliases to the same object.


In [49]:
data = {
    "a": 1,
    "b": 2,
    "c": 3,
}

current = data
other = data

print("Same object:", current is other)

assert current is other


Same object: True


## Why variable names do not protect us

Mutation safety is about **object identity**, not variable spelling.

If `current is other`, then mutating `other` structurally mutates the very object being iterated through `current`.


## Safe repair

Freeze the key domain before deleting.


In [50]:
for key in list(current):
    other.pop(key)

print(data)

assert data == {}


{}


# Problem 29 — Encapsulated Mutation Can Still Break Iteration

## Situation

The structural mutation is hidden inside a helper:

```python
for name in users:
    remove_if_disabled(users, name)
```

This can make the problem harder to notice in code review.

We will redesign the responsibility boundaries.


## Step 1 — Make the discovery function read-only


In [51]:
users = {
    "alice": {"enabled": True},
    "bob": {"enabled": False},
    "carol": {"enabled": True},
    "dave": {"enabled": False},
}


def disabled_names(
    users: Mapping[str, Mapping[str, Any]],
) -> list[str]:
    return [
        name
        for name, record in users.items()
        if record.get("enabled") is False
    ]


to_remove = disabled_names(users)
print(to_remove)


['bob', 'dave']


## Step 2 — Perform mutation at the caller level


In [52]:
for name in to_remove:
    del users[name]

print(users)

assert set(users) == {"alice", "carol"}


{'alice': {'enabled': True}, 'carol': {'enabled': True}}


## Takeaway

A helper that *looks* like a predicate or inspection function should not unexpectedly resize the collection its caller is iterating.

Separating reads from structural writes improves predictability.


# Problem 30 — Rebuild or Mutate?

## Situation

We want to retain only dictionary entries whose score is at least 70.

Two valid designs are possible:

1. delete unwanted entries from the original,
2. build a new filtered mapping.

Let's compare their semantics.


In [53]:
scores = {
    "Ada": 91,
    "Linus": 68,
    "Grace": 95,
    "Guido": 72,
    "Barbara": 64,
}

filtered = {
    name: score
    for name, score in scores.items()
    if score >= 70
}

print("Original:", scores)
print("Filtered:", filtered)


Original: {'Ada': 91, 'Linus': 68, 'Grace': 95, 'Guido': 72, 'Barbara': 64}
Filtered: {'Ada': 91, 'Grace': 95, 'Guido': 72}


## Interpretation

The comprehension creates a new dictionary and leaves the original untouched.

That may be preferable when:

- callers may still need the original,
- the transformation is conceptually functional,
- rollback or comparison is useful.


## Alternative: mutate the same object

If preserving dictionary identity is required, stage the result, then commit.


In [54]:
scores.clear()
scores.update(filtered)

print(scores)

assert list(scores) == ["Ada", "Grace", "Guido"]


{'Ada': 91, 'Grace': 95, 'Guido': 72}


# Problem 31 — Reusing a Long-Lived View

## Situation

Because a view is live, we can store it once and reuse it.

Let's compare:

```python
for key in d.keys():
```

with:

```python
keys = d.keys()
for key in keys:
```

in repeated loops.


In [55]:
d = {
    i: i
    for i in range(5000)
}

keys = d.keys()

def recreate_view() -> None:
    for _ in range(30):
        for key in d.keys():
            pass


def reuse_view() -> None:
    for _ in range(30):
        for key in keys:
            pass


t1 = timeit(recreate_view, number=100)
t2 = timeit(reuse_view, number=100)

print(f"Recreate: {t1:.4f}s")
print(f"Reuse:    {t2:.4f}s")


Recreate: 0.2421s
Reuse:    0.2281s


## Interpretation

The timing difference is usually much less important than program structure.

Do not cache a view merely because it might save a tiny amount of overhead.

Cache it when its **live-reference semantics** are useful.


# Problem 32 — A Key-Domain Watcher

## Situation

Create an object that stores a key view and can later answer:

- how many keys exist,
- whether a particular key currently exists,
- which keys are currently visible.

The dictionary itself will change after the watcher is created.


In [56]:
class KeyDomainWatcher:
    def __init__(self, mapping: dict[str, Any]) -> None:
        self._keys = mapping.keys()

    def count(self) -> int:
        return len(self._keys)

    def contains(self, key: str) -> bool:
        return key in self._keys

    def snapshot(self) -> tuple[str, ...]:
        return tuple(self._keys)


registry = {
    "alpha": 1,
    "beta": 2,
}

watcher = KeyDomainWatcher(registry)

registry["gamma"] = 3
del registry["alpha"]

print(watcher.snapshot())

assert watcher.contains("gamma")
assert not watcher.contains("alpha")
assert watcher.count() == 2


('beta', 'gamma')


## Design lesson

A live view can be intentionally stored inside a longer-lived object.

This is not automatically a memory or design problem; it is a semantic choice.

The important part is to document that the object reports **current**, not historical, state.


# Problem 33 — Membership Semantics Across Views

## Situation

Dictionary syntax supports several kinds of membership tests.

Let's compare them explicitly.


In [57]:
d = {
    "x": 10,
    "y": 20,
}

checks = [
    ('"x" in d', "x" in d),
    ('"x" in d.keys()', "x" in d.keys()),
    ('10 in d.values()', 10 in d.values()),
    ('("x", 10) in d.items()', ("x", 10) in d.items()),
    ('("x", 99) in d.items()', ("x", 99) in d.items()),
]

for expression, result in checks:
    print(f"{expression:<28} -> {result}")


"x" in d                     -> True
"x" in d.keys()              -> True
10 in d.values()             -> True
("x", 10) in d.items()       -> True
("x", 99) in d.items()       -> False


## Style guidance

For plain key membership, prefer:

```python
if key in d:
```

It is idiomatic and concise.

Use `.keys()` explicitly when the view itself adds meaning, such as in set algebra.


# Problem 34 — Compare Key Equality and Item Equality

## Situation

Two dictionaries may have:

- the same keys but different values,
- the same exact items but a different insertion order.

Let's separate those ideas.


In [58]:
a = {
    "x": 1,
    "y": 2,
}

b = {
    "x": 100,
    "y": 200,
}

c = {
    "y": 2,
    "x": 1,
}

print("a.keys() == b.keys():", a.keys() == b.keys())
print("a.items() == b.items():", a.items() == b.items())
print("a.items() == c.items():", a.items() == c.items())

assert a.keys() == b.keys()
assert a.items() != b.items()
assert a.items() == c.items()


a.keys() == b.keys(): True
a.items() == b.items(): False
a.items() == c.items(): True


## Takeaway

Key-view equality asks whether the dictionaries have the same key domain.

Item-view equality asks whether they have the same exact key/value pairs.

Neither comparison is about insertion order.


# Problem 35 — Merge Defaults Only for Missing Keys

## Situation

We have application defaults and user configuration.

Existing user choices must remain untouched.

We want to know which defaults are missing *before* applying them.


In [59]:
defaults = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "workers": 4,
}

config = {
    "host": "api.internal",
    "debug": True,
}

missing = defaults.keys() - config.keys()

print("Missing keys:", missing)


Missing keys: {'port', 'workers'}


## Step 2 — Apply only those keys


In [60]:
for key in missing:
    config[key] = defaults[key]

print(config)

assert config["host"] == "api.internal"
assert config["debug"] is True
assert config["port"] == 8000
assert config["workers"] == 4


{'host': 'api.internal', 'debug': True, 'port': 8000, 'workers': 4}


## Design note

Other dictionary APIs can solve the merge itself.

The view-based approach is especially useful when the **missing-key set is also valuable information** for logging, validation, or reporting.


# Problem 36 — Stable Batch Processing vs Live Processing

## Situation

A queue dictionary receives new entries over time.

A particular batch must process exactly the jobs that existed when the batch started.

A live view is therefore the wrong abstraction for batch membership.


## Step 1 — Freeze the batch


In [61]:
queue = {
    "job-1": "compile",
    "job-2": "test",
}

batch = tuple(queue.items())

# New work arrives after the batch starts.
queue["job-3"] = "deploy"

print("Batch:", batch)
print("Current queue:", queue)


Batch: (('job-1', 'compile'), ('job-2', 'test'))
Current queue: {'job-1': 'compile', 'job-2': 'test', 'job-3': 'deploy'}


## Step 2 — Process only the frozen batch


In [62]:
processed = []

for job_id, action in batch:
    processed.append((job_id, action.upper()))

print(processed)

assert [job_id for job_id, _ in processed] == [
    "job-1",
    "job-2",
]


[('job-1', 'COMPILE'), ('job-2', 'TEST')]


## Takeaway

A live view is ideal for dashboards and observers.

A snapshot is ideal for batch boundaries.

Same dictionary, different semantics.


# Problem 37 — Defensive `delete_where`

## Situation

We want a reusable helper that deletes entries matching a predicate.

The helper should not expose the caller to iterator invalidation.


## Step 1 — Freeze only the keys selected for deletion


In [63]:
def delete_where(
    d: dict[Any, Any],
    predicate: Callable[[Any, Any], bool],
) -> int:
    selected = [
        key
        for key, value in d.items()
        if predicate(key, value)
    ]

    for key in selected:
        del d[key]

    return len(selected)


## Step 2 — Test it on a realistic mapping


In [64]:
temperatures = {
    "sensor-A": 18.2,
    "sensor-B": -999.0,
    "sensor-C": 21.1,
    "sensor-D": -999.0,
}

deleted = delete_where(
    temperatures,
    lambda key, value: value == -999.0,
)

print("Deleted:", deleted)
print("Remaining:", temperatures)

assert deleted == 2
assert set(temperatures) == {"sensor-A", "sensor-C"}


Deleted: 2
Remaining: {'sensor-A': 18.2, 'sensor-C': 21.1}


## Takeaway

The helper's contract now guarantees safe structural mutation internally.

Callers do not need to remember the iterator rule each time.


# Problem 38 — In-Place Value Transformation Helper

## Situation

Structural mutation is unsafe during normal iteration, but replacing existing values is a legitimate operation.

Let's encapsulate that safe pattern.


In [65]:
def transform_values_in_place(
    d: dict[Any, Any],
    transform: Callable[[Any, Any], Any],
) -> None:
    for key, value in d.items():
        d[key] = transform(key, value)


## Example — Apply a service-specific adjustment


In [66]:
latencies = {
    "search": 100.0,
    "upload": 250.0,
    "billing": 80.0,
}

transform_values_in_place(
    latencies,
    lambda service, value: (
        value * 1.10
        if service == "upload"
        else value
    ),
)

print(latencies)

assert latencies["upload"] == 275.0


{'search': 100.0, 'upload': 275.0, 'billing': 80.0}


## Why this helper is safe

Its contract never changes the key domain.

That is a stronger guarantee than accepting an arbitrary callback that may insert or delete entries.


# Problem 39 — What Should an API Return: View or Snapshot?

## Situation

We are designing:

```python
def registered_names(...):
    ...
```

Should it return:

```python
registry.keys()
```

or:

```python
tuple(registry)
```

There is no universally correct answer.

We must decide the API semantics.


## Option A — Return a live view

Advantages:

- reflects later changes,
- avoids copying keys,
- supports set-like operations.

Risks:

- callers may accidentally retain the registry,
- callers may assume historical stability,
- results can change between observations.


## Option B — Return a snapshot

Advantages:

- stable result,
- easier historical reasoning,
- independent batch semantics.

Costs:

- requires allocating a new container,
- does not reflect later updates.


In [67]:
class ServiceCatalog:
    def __init__(self) -> None:
        self._services: dict[str, Any] = {}

    def register(self, name: str, service: Any) -> None:
        self._services[name] = service

    def live_names(self):
        return self._services.keys()

    def snapshot_names(self) -> tuple[str, ...]:
        return tuple(self._services)


catalog = ServiceCatalog()
catalog.register("search", object())

live = catalog.live_names()
snapshot = catalog.snapshot_names()

catalog.register("upload", object())

print("Live:", tuple(live))
print("Snapshot:", snapshot)


Live: ('search', 'upload')
Snapshot: ('search',)


## Takeaway

The choice between a view and a snapshot belongs in API design documentation because it determines how the returned object behaves over time.


# Problem 40 — Final Integrated Challenge: Reconcile Two Registries

## Situation

Two registries describe workers and their assigned queues.

We need to produce a reconciliation report with:

- workers only in the old registry,
- workers only in the new registry,
- workers whose assignment is unchanged,
- workers whose assignment changed.

Then we need to safely mutate the old dictionary so it becomes exactly the new dictionary.

We will solve this in stages.


## Step 1 — Define the registries


In [68]:
old_registry = {
    "worker-1": "queue-A",
    "worker-2": "queue-B",
    "worker-3": "queue-C",
    "worker-5": "queue-E",
}

new_registry = {
    "worker-1": "queue-A",
    "worker-2": "queue-X",
    "worker-4": "queue-D",
    "worker-5": "queue-E",
}


## Step 2 — Compare key domains


In [69]:
old_keys = old_registry.keys()
new_keys = new_registry.keys()

removed_workers = old_keys - new_keys
added_workers = new_keys - old_keys
common_workers = old_keys & new_keys

print("Removed:", removed_workers)
print("Added:", added_workers)
print("Common:", common_workers)


Removed: {'worker-3'}
Added: {'worker-4'}
Common: {'worker-2', 'worker-1', 'worker-5'}


## Step 3 — Split common workers by assignment


In [70]:
unchanged_workers = {
    worker
    for worker in common_workers
    if old_registry[worker] == new_registry[worker]
}

changed_workers = {
    worker
    for worker in common_workers
    if old_registry[worker] != new_registry[worker]
}

print("Unchanged:", unchanged_workers)
print("Changed:", changed_workers)


Unchanged: {'worker-1', 'worker-5'}
Changed: {'worker-2'}


## Step 4 — Build the report


In [71]:
report = {
    "removed": {
        worker: old_registry[worker]
        for worker in removed_workers
    },
    "added": {
        worker: new_registry[worker]
        for worker in added_workers
    },
    "changed": {
        worker: (
            old_registry[worker],
            new_registry[worker],
        )
        for worker in changed_workers
    },
    "unchanged": {
        worker: old_registry[worker]
        for worker in unchanged_workers
    },
}

print(report)


{'removed': {'worker-3': 'queue-C'}, 'added': {'worker-4': 'queue-D'}, 'changed': {'worker-2': ('queue-B', 'queue-X')}, 'unchanged': {'worker-1': 'queue-A', 'worker-5': 'queue-E'}}


## Step 5 — Safely commit the new registry

We already have the complete target state.

There is no need to perform fragile individual key mutations while iterating.

Replace the contents in two explicit operations.


In [72]:
old_registry.clear()
old_registry.update(new_registry)

print(old_registry)

assert old_registry == new_registry


{'worker-1': 'queue-A', 'worker-2': 'queue-X', 'worker-4': 'queue-D', 'worker-5': 'queue-E'}


# Final Review

The central lesson of dictionary views is not merely that they are “dynamic.”

The deeper lesson is that they force us to reason about **time**, **ownership**, and **mutation boundaries**.

## Use a live view when

- future dictionary changes should remain visible,
- set-like key operations are useful,
- you want direct iteration over current items or values,
- a live observer is semantically correct.

## Use a snapshot when

- a batch must remain stable,
- historical state matters,
- structural mutation will happen during processing,
- you want to decouple the result from future dictionary changes.

## Structural mutation checklist

Before adding, deleting, or renaming keys inside logic related to iteration, ask:

1. Am I currently iterating this same dictionary?
2. Could an alias refer to the same dictionary?
3. Could a helper function resize it?
4. Should I collect keys first?
5. Would rebuilding a new dictionary be simpler?
6. Should I stage and commit transactionally?

## Iteration style checklist

Use:

```python
for key in d:
```

when only keys are needed.

Use:

```python
for key, value in d.items():
```

when both are needed.

Use:

```python
d.keys()
```

when the key **view itself** matters.

Use:

```python
list(d)
tuple(d.items())
```

when stable processing is required.

## Final mental model

A dictionary view is best imagined as a **window into the mapping**, not a container filled with copied data.

That single mental model explains:

- why views update,
- why snapshots behave differently,
- why structural mutation is dangerous during iteration,
- why key views support set-like reasoning,
- why a long-lived view can be useful in APIs,
- and why choosing between live and stable state is a design decision.
